# BFND step 2: controls, bootstrap CI and evidence sheet

Run this notebook from a fresh runtime with **Runtime → Run all**.

**Before you start, put these in your Google Drive `MyDrive`:**
- `BFND_step2_code.zip` (the current code: new controls, bootstrap, evidence sheet, tests)
- `BFND-data/` folder with `features/` (must contain `meta.csv`, `bert.npy`, `clip_img.npy`), and `results/selection/` from the earlier model-selection run
- (for the evidence sheet only) `BFND-data/data/evidence/evidence.csv`

**What it does, in order:**
1. Mounts Drive and checks the folders.
2. Unzips the code into the Colab runtime.
3. Checks that the data files exist.
4. Runs the new tests.
5. Runs the confound controls (script 10).
6. Runs the paired bootstrap CI (script 12).
7. Builds the new 240-row evidence sheet (script 11), only if it does not exist yet.
8. Copies the outputs into `MyDrive/BFND-data/exports/<time>/`. Nothing is deleted.

**Notes:**
- A GPU is optional. Choose **Runtime → Change runtime type → T4 GPU** if you like; the heads are small and CPU also works.
- If a cell fails, read the message, fix the cause, then **Runtime → Restart session** and **Run all** again.
- The evidence sheet is **not** labelled by this notebook. Labelling is a human step (see the last section).

## Cell 1: mount Drive and check the top-level folders

In [1]:
# Mount Google Drive and check the folders this notebook needs
import os
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive"
BFND = f"{DRIVE}/BFND-data"
CODE_ZIP = f"{DRIVE}/BFND_step2_code.zip"

assert os.path.exists(CODE_ZIP), f"Upload BFND_step2_code.zip to MyDrive first. Not found: {CODE_ZIP}"
assert os.path.isdir(BFND), f"BFND-data folder not found in MyDrive: {BFND}"
assert os.path.isdir(f"{BFND}/features"), "BFND-data/features is missing. The earlier feature run must be finished."
print("Drive mounted. Code zip and BFND-data found.")

Mounted at /content/drive
Drive mounted. Code zip and BFND-data found.


## Cell 2: install the test runner

In [2]:
# pytest is needed for the tests. Colab usually has pandas, numpy, scikit-learn and torch already.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print("pytest ready")

pytest ready


## Cell 3: unzip the code into the runtime

In [3]:
# Unzip into a runtime folder (not Drive). Existing files are overwritten; nothing is deleted.
import os, zipfile

EXTRACT = "/content/bfnd_code"
os.makedirs(EXTRACT, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(EXTRACT)

# The zip may hold one top-level folder. Find the folder that contains the new files.
PROJECT = None
for root, dirs, files in os.walk(EXTRACT):
    if "src" in dirs and "scripts" in dirs and "configs" in dirs:
        PROJECT = root
        break
assert PROJECT is not None, "Could not find src/, scripts/ and configs/ inside the zip"

required = [
    "src/confound_controls.py", "scripts/10_confound_controls.py",
    "src/bootstrap.py", "scripts/12_bootstrap_ci.py",
    "src/evidence_eval.py", "scripts/11_evidence_eval_sheet.py",
    "tests/test_confound_controls.py", "tests/test_evidence_eval.py", "tests/test_bootstrap.py",
]
missing = [f for f in required if not os.path.exists(os.path.join(PROJECT, f))]
assert not missing, f"These files are missing from the zip: {missing}"
print("Project folder:", PROJECT)
print("All new files found:", len(required))

Project folder: /content/bfnd_code/BFND_code
All new files found: 9


## Cell 4: check the data files

In [4]:
# Check the feature files and show what is inside meta.csv.
# The script stops here if something is wrong, so the later cells are not run on bad input.
import os
import numpy as np
import pandas as pd

FEATURES = f"{BFND}/features"
for f in ["meta.csv", "bert.npy", "clip_img.npy"]:
    assert os.path.exists(f"{FEATURES}/{f}"), f"Missing feature file: {FEATURES}/{f}"

meta = pd.read_csv(f"{FEATURES}/meta.csv")
for col in ["sample_id", "split", "y", "has_image", "model_text"]:
    assert col in meta.columns, f"meta.csv has no column '{col}'"
print("meta rows:", len(meta))
print(meta["split"].value_counts().to_string())
print("has_image share:", round(meta["has_image"].astype(bool).mean(), 3))
print("bert shape:", np.load(f"{FEATURES}/bert.npy").shape)
print("clip shape:", np.load(f"{FEATURES}/clip_img.npy").shape)

EVIDENCE_CSV = f"{BFND}/data/evidence/evidence.csv"
print("evidence.csv found:", os.path.exists(EVIDENCE_CSV), "(needed only for cell 8)")

meta rows: 3910
split
train    2736
test      588
val       586
has_image share: 0.613
bert shape: (3910, 768)
clip shape: (3910, 512)
evidence.csv found: True (needed only for cell 8)


## Cell 5: run the new tests

In [5]:
# Tests for the new code. They use synthetic data and do not need the features.
# check=True: if a test fails, this cell stops with an error.
import subprocess, sys

subprocess.run(
    [sys.executable, "-m", "pytest", "-q",
     "tests/test_confound_controls.py", "tests/test_evidence_eval.py", "tests/test_bootstrap.py"],
    cwd=PROJECT, check=True,
)
print("New tests passed")

New tests passed


## Cell 6: confound controls (script 10)

In [6]:
# Image-content control, presence-only control, length quartiles and near-duplicate audit.
# Writes results/controls/controls.json and controls.md in BFND-data.
import subprocess, sys

subprocess.run([sys.executable, "scripts/10_confound_controls.py"], cwd=PROJECT, check=True)

print("\n----- controls.md -----")
print(open(f"{BFND}/results/controls/controls.md", encoding="utf-8").read())


----- controls.md -----
# Confound controls (diagnostic, validation used for reading)

| Configuration | Val macro-F1 (mean ± std) |
|---|---|
| text_banglabert | 0.7021 ± 0.0044 |
| presence_only | 0.1789 ± 0.0000 |
| length_only | 0.4582 ± 0.0095 |
| text_plus_presence | 0.7074 ± 0.0026 |
| fusion_real | 0.7463 ± 0.0035 |
| fusion_image_shuffled | 0.6785 ± 0.0084 |

## Contrasts (validation macro-F1)
- image_content_beyond_presence (fusion_real - text_plus_presence): +0.0389
- shuffled_image_beyond_presence (fusion_image_shuffled - text_plus_presence): -0.0289
- presence_gain_on_text (text_plus_presence - text_banglabert): +0.0053

## Near-duplicates across splits (threshold 0.8)
- val: 4 of 586 near-match a train post; 0 exact
- test: 1 of 588 near-match a train post; 0 exact



## Cell 7: paired bootstrap CI (script 12)

This compares two saved models on the same test rows. The names must match the files in
`BFND-data/results/selection/probs/`. The defaults below are the usual ones; change them if your
names differ. The first run prints the available names.

In [7]:
# Choose the two models to compare: A is the baseline, B is the model you want to test.
# The names are the same as in script 08, with ':' and '+' replaced by '_'.
import os, subprocess, sys

A_NAME = "text_banglabert"
B_NAME = "fusion_early_banglabert_clip"

PROBS = f"{BFND}/results/selection/probs"
assert os.path.isdir(PROBS), "results/selection/probs not found. Run the model-selection step first."

print("Available names in probs/:")
subprocess.run([sys.executable, "scripts/12_bootstrap_ci.py"], cwd=PROJECT, check=True)

def has_all_seeds(name):
    return all(os.path.exists(f"{PROBS}/{name}_seed{s}.npy") for s in [42, 43, 44])

if has_all_seeds(A_NAME) and has_all_seeds(B_NAME):
    subprocess.run([sys.executable, "scripts/12_bootstrap_ci.py", "--a", A_NAME, "--b", B_NAME],
                   cwd=PROJECT, check=True)
else:
    print("Skipped: change A_NAME / B_NAME to names listed above (all 3 seeds must exist).")

Available names in probs/:


## Cell 8: build the 240-row evidence sheet (script 11)

This makes the sheet only if it does not exist yet. The script never overwrites a sheet, so labels
that a person has already filled are safe.

In [8]:
# Build the new evidence relevance sheet with a frozen dev/eval split.
# The sheet is for a human to label. Nothing is labelled here.
import os, subprocess, sys

SHEET = f"{BFND}/data/annotations/evidence_eval_sheet.csv"
ROLES = f"{BFND}/data/annotations/evidence_eval_roles.json"

if os.path.exists(SHEET):
    print("Sheet already exists, so it was not rebuilt:", SHEET)
    print("Open it and fill relevant_0_or_1 with 0 or 1.")
elif not os.path.exists(f"{BFND}/data/evidence/evidence.csv"):
    print("Skipped: evidence.csv is not in BFND-data/data/evidence/. Add it, then Run all again.")
else:
    subprocess.run([sys.executable, "scripts/11_evidence_eval_sheet.py"], cwd=PROJECT, check=True)
    import pandas as pd
    s = pd.read_csv(SHEET)
    print("Rows:", len(s))
    print(s.groupby("role")["post_id"].nunique().to_string())
    print("Unlabelled rows:", int(s["relevant_0_or_1"].isna().sum()))

Rows: 240
role
dev     40
eval    40
Unlabelled rows: 240


## Cell 9: copy the outputs to Drive (copy only, nothing is deleted)

In [9]:
# Copy results and the sheet to MyDrive/BFND-data/exports/<time>/ so they survive the runtime.
import os, shutil, time

stamp = time.strftime("%Y%m%d_%H%M")
EXPORT = f"{BFND}/exports/{stamp}"
os.makedirs(EXPORT, exist_ok=True)

copied = []
ctrl = f"{BFND}/results/controls"
if os.path.isdir(ctrl):
    shutil.copytree(ctrl, f"{EXPORT}/controls", dirs_exist_ok=True)
    copied.append("controls/")
for src in [f"{BFND}/results/selection/bootstrap_ci.json",
            f"{BFND}/data/annotations/evidence_eval_sheet.csv",
            f"{BFND}/data/annotations/evidence_eval_roles.json"]:
    if os.path.exists(src):
        shutil.copy(src, EXPORT)
        copied.append(os.path.basename(src))

print("Exported to:", EXPORT)
print("Copied:", copied if copied else "nothing yet (earlier cells were skipped)")

Exported to: /content/drive/MyDrive/BFND-data/exports/20261125_1812
Copied: ['controls/', 'bootstrap_ci.json', 'evidence_eval_sheet.csv', 'evidence_eval_roles.json']


## Cell 10: what to send back

- From cell 6: the `controls.md` table and the three contrast lines.
- From cell 7: the bootstrap CI lines (macro-F1 and Satire F1).
- From cell 8: the row count and the unlabelled count.

**Labelling (a person must do this):** open `evidence_eval_sheet.csv` in Drive.
For each row, read `post_text` and `passage`. Put `1` in `relevant_0_or_1` if the passage is
about the claim in the post, otherwise `0`. Label the rows with `role = dev` first. Do not change
`role`, `post_id` or `passage`. Do not label the `eval` rows until the dev analysis is done.